In [14]:
import os

# 1. Get the absolute path of the project root (one level up from /dataset)
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
cache_path = os.path.join(project_root, "hf_cache")

# 2. Set environment variables (Must be done BEFORE importing transformers)
os.environ['HF_HOME'] = cache_path
os.environ['HF_HUB_CACHE'] = os.path.join(cache_path, "hub")

# 3. Verify the path and list models
if os.path.exists(os.environ['HF_HUB_CACHE']):
    print(f"✅ Success! Cache found at: {os.environ['HF_HUB_CACHE']}")
    # List model folders (e.g., models--Qwen--Qwen2.5-0.5B-Instruct)
    models = [d for d in os.listdir(os.environ['HF_HUB_CACHE']) if d.startswith("models--")]
    print("\nAvailable Cached Models:")
    for m in models:
        print(f" - {m.replace('models--', '').replace('--', '/')}")
else:
    print(f"❌ Error: Path not found at {os.environ['HF_HUB_CACHE']}")
    print("Check if 'hf_cache' is spelled correctly in your root folder.")

✅ Success! Cache found at: /d/hpc/projects/onj_fri/no-language-processors-v2/hf_cache/hub

Available Cached Models:
 - Qwen/Qwen2.5-7B-Instruct


In [11]:
import os
hub_path = os.environ['HF_HUB_CACHE']
model_folder = os.path.join(hub_path, "models--Qwen--Qwen2.5-7B-Instruct", "snapshots")

if os.path.exists(model_folder):
    # Get the name of the long alphanumeric folder inside snapshots
    snapshots = os.listdir(model_folder)
    if snapshots:
        actual_model_path = os.path.join(model_folder, snapshots[0])
        print(f"✅ Found weights at: {actual_model_path}")
    else:
        print("❌ Snapshots folder is empty.")
else:
    print(f"❌ Could not find snapshot folder at {model_folder}")

✅ Found weights at: /d/hpc/projects/onj_fri/no-language-processors-v2/hf_cache/hub/models--Qwen--Qwen2.5-7B-Instruct/snapshots/a09a35458c702b33eeacc393d103063234e8bc28


In [18]:
%pip install --upgrade transformers huggingface_hub accelerate

Note: you may need to restart the kernel to use updated packages.


In [6]:
import os

# Define the base model folder
base_model_dir = "/d/hpc/projects/onj_fri/no-language-processors-v2/hf_cache/hub/models--Qwen--Qwen2.5-7B-Instruct"

# Walk through the directory to find the actual config.json
for root, dirs, files in os.walk(base_model_dir):
    if "config.json" in files:
        print(f"Found it here: {os.path.join(root, 'config.json')}")
    else: 
        print(f"NOT FOUND")

NOT FOUND
NOT FOUND
NOT FOUND
NOT FOUND
NOT FOUND


In [7]:
from transformers import Qwen2ForCausalLM, AutoTokenizer, Qwen2Config
import torch

# This will download the model to your current folder instead of the shared cache
# The 0.5B version is tiny (1GB), fast, and great for testing this automation!
model_id = "Qwen/Qwen2.5-0.5B-Instruct" 

print("Downloading model to local folder...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("🚀 Local model ready!")

OSError: [Errno 28] No space left on device

In [9]:
from transformers import Qwen2ForCausalLM, AutoTokenizer, Qwen2Config
import torch

# 1. The snapshot path where the weights (.safetensors files) are located
model_path = "/d/hpc/projects/onj_fri/no-language-processors-v2/hf_cache/hub/models--Qwen--Qwen2.5-7B-Instruct/snapshots/a09a35458c702b33eeacc393d103063234e8bc28"

print("Step 1: Manually defining the architecture config...")
# These are the exact specs for Qwen2.5-7B
config = Qwen2Config(
    vocab_size=152064,
    hidden_size=3584,
    intermediate_size=18944,
    num_hidden_layers=28,
    num_attention_heads=28,
    num_key_value_heads=4,
    hidden_act="silu",
    max_position_embeddings=32768,
    initializer_range=0.02,
    rms_norm_eps=1e-6,
    use_cache=True,
    tie_word_embeddings=False,
    rope_theta=1000000.0,
)

print("Step 2: Loading tokenizer (ignoring local config)...")
# Tokenizers are usually more flexible; if this fails, we can use the 0.5B tokenizer
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-7B-Instruct", trust_remote_code=True)

print("Step 3: Loading weights into the manual config (this will take time)...")
model = Qwen2ForCausalLM.from_pretrained(
    model_path,
    config=config,
    torch_dtype=torch.float16,
    device_map="auto",
    local_files_only=True
)

print("🚀 Success! Model loaded by manually defining the architecture.")

Step 1: Manually defining the architecture config...
Step 2: Loading tokenizer (ignoring local config)...


/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:805: UserWarning: Not enough free disk space to download the file. The expected file size is: 0.01 MB. The target location /scratch/hf_cache/models--Qwen--Qwen2.5-7B-Instruct/blobs only has 0.00 MB free disk space.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

OSError: [Errno 28] No space left on device

In [10]:
# This checks disk space for the specific partitions
!df -h /d/hpc/projects/onj_fri/
!df -h /scratch/

Filesystem                                                                                                                                                                                                           Size  Used Avail Use% Mounted on
[2001:1470:8000:406::ce:11],[153.5.73.41],[2001:1470:8000:406::ce:12],[153.5.73.42],[2001:1470:8000:406::ce:13],[153.5.73.43],[2001:1470:8000:406::ce:14],[153.5.73.44],[2001:1470:8000:406::ce:15],[153.5.73.45]:/  7.1P  783T  6.3P  11% /d/hpc/projects/onj_fri
Filesystem      Size  Used Avail Use% Mounted on
fuse-overlayfs   16M   16M     0 100% /


In [11]:
# Check size of your project and cache folders
!du -sh /d/hpc/projects/onj_fri/no-language-processors-v2/*

512	/d/hpc/projects/onj_fri/no-language-processors-v2/Untitled.ipynb
93K	/d/hpc/projects/onj_fri/no-language-processors-v2/celestial_sphere.html
178K	/d/hpc/projects/onj_fri/no-language-processors-v2/dataset
222K	/d/hpc/projects/onj_fri/no-language-processors-v2/gaia_report.html
15G	/d/hpc/projects/onj_fri/no-language-processors-v2/hf_cache
429K	/d/hpc/projects/onj_fri/no-language-processors-v2/logs
15G	/d/hpc/projects/onj_fri/no-language-processors-v2/old_sif_and_sh 
512	/d/hpc/projects/onj_fri/no-language-processors-v2/outputs
414M	/d/hpc/projects/onj_fri/no-language-processors-v2/py_packages
686K	/d/hpc/projects/onj_fri/no-language-processors-v2/src
0	/d/hpc/projects/onj_fri/no-language-processors-v2/untitled.py
6.5K	/d/hpc/projects/onj_fri/no-language-processors-v2/vllm.def
1.5K	/d/hpc/projects/onj_fri/no-language-processors-v2/vllm.sh
7.6G	/d/hpc/projects/onj_fri/no-language-processors-v2/vllm.sif


In [12]:
import os
import shutil

cache_dir = "/d/hpc/projects/onj_fri/no-language-processors-v2/hf_cache"

# Remove incomplete downloads
!find {cache_dir} -name "*.incomplete" -delete

print("Cleaned up incomplete download files.")

Cleaned up incomplete download files.


In [18]:
!huggingface-cli scan-cache

⚠️  Warning: 'huggingface-cli scan-cache' is deprecated. Use 'hf cache scan' instead.
Cache directory not found: /scratch/hf_cache/hub


In [19]:
import os

# Define your spacious project directory
project_dir = "/d/hpc/projects/onj_fri/no-language-processors-v2"
new_temp_dir = os.path.join(project_dir, "tmp")

# Create the temp folder if it doesn't exist
os.makedirs(new_temp_dir, exist_ok=True)

# Force ALL temporary systems to use your project space
os.environ['TMPDIR'] = new_temp_dir
os.environ['TEMP'] = new_temp_dir
os.environ['TMP'] = new_temp_dir

# Redirect Hugging Face Home and Cache
os.environ['HF_HOME'] = os.path.join(project_dir, "hf_cache")
os.environ['HF_DATASETS_CACHE'] = os.path.join(project_dir, "hf_cache", "datasets")

print(f"✅ All temporary traffic redirected to: {new_temp_dir}")

✅ All temporary traffic redirected to: /d/hpc/projects/onj_fri/no-language-processors-v2/tmp


In [22]:
from transformers import Qwen2TokenizerFast, Qwen2ForCausalLM
import torch

# Verified path for the heavy weights
model_path = "/d/hpc/projects/onj_fri/no-language-processors-v2/hf_cache/hub/models--Qwen--Qwen2.5-7B-Instruct/snapshots/a09a35458c702b33eeacc393d103063234e8bc28"

print("Step 1: Loading tokenizer using compatible Qwen2.5-0.5B logic...")
# This avoids the broken local files and uses the redirected TMPDIR in your project folder
tokenizer = Qwen2TokenizerFast.from_pretrained(
    "Qwen/Qwen2.5-0.5B-Instruct", 
    trust_remote_code=True
)

print("Step 2: Loading 7B model weights using manual config...")
# Using the 'config' object we manually built to bypass the AutoConfig error
model = Qwen2ForCausalLM.from_pretrained(
    model_path,
    config=config, 
    torch_dtype=torch.float16,
    device_map="auto",
    local_files_only=True
)

print("🚀 SUCCESS: Qwen2.5-7B is loaded and ready for ADQL generation!")

Step 1: Loading tokenizer using compatible Qwen2.5-0.5B logic...


/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:805: UserWarning: Not enough free disk space to download the file. The expected file size is: 0.01 MB. The target location /scratch/hf_cache/models--Qwen--Qwen2.5-0.5B-Instruct/blobs only has 0.00 MB free disk space.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

OSError: [Errno 28] No space left on device

In [23]:
from transformers import Qwen2TokenizerFast, Qwen2ForCausalLM
import torch
import os

# 1. Define your safe project space
project_hf_cache = "/d/hpc/projects/onj_fri/no-language-processors-v2/hf_cache"
model_path = os.path.join(project_hf_cache, "hub/models--Qwen--Qwen2.5-7B-Instruct/snapshots/a09a35458c702b33eeacc393d103063234e8bc28")

print("Step 1: Loading tokenizer with EXPLICIT cache redirection...")
# We explicitly pass cache_dir to force it away from /scratch
tokenizer = Qwen2TokenizerFast.from_pretrained(
    "Qwen/Qwen2.5-0.5B-Instruct", 
    cache_dir=project_hf_cache,
    trust_remote_code=True
)

print("Step 2: Loading 7B weights from local storage...")
model = Qwen2ForCausalLM.from_pretrained(
    model_path,
    config=config, # Uses the manual config we built earlier
    torch_dtype=torch.float16,
    device_map="auto",
    local_files_only=True
)

print("🚀 Finally loaded! No files were written to the full /scratch partition.")

Step 1: Loading tokenizer with EXPLICIT cache redirection...


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


Step 2: Loading 7B weights from local storage...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

🚀 Finally loaded! No files were written to the full /scratch partition.


In [25]:
import pandas as pd
import torch

def generate_gaia_queries(complexity, count=5):
    # Professional prompt for Gaia ADQL/SQL
    prompt = f"""Task: Generate {count} unique {complexity} level ADQL (SQL) queries for the Gaia database.
    Target Table: gaiadr3.gaia_source
    Common Columns: ra, dec, parallax, phot_g_mean_mag, pmra, pmdec.
    
    Format each line exactly like this:
    SQL: [query] | NL: [natural language question]
    
    New {complexity} Queries:"""

    # Format for the model's chat template
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=1024,
        do_sample=True,
        temperature=0.7
    )
    
    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    # Extract only the new lines generated after the prompt
    lines = response.split(f"New {complexity} Queries:")[-1].strip().split('\n')
    return [l for l in lines if "|" in l]

# --- THE AUTOMATION LOOP ---
total_needed = 100
results = []
output_file = "dataset/gaia_1000_queries.csv"

print(f"Starting generation of {total_needed} queries...")

for i in range(0, total_needed, 5):
    # Cycle through complexity levels
    level = ["Simple", "Medium", "Complex"][(i // 5) % 3]
    
    try:
        batch = generate_gaia_queries(level, count=5)
        for entry in batch:
            sql, nl = entry.split("|")
            results.append({
                "complexity": level,
                "sql": sql.replace("SQL:", "").strip(),
                "question": nl.replace("NL:", "").strip()
            })
        
        # Save progress every 50 queries so you don't lose data
        if len(results) % 50 == 0:
            pd.DataFrame(results).to_csv(output_file, index=False)
            print(f"✅ Progress: {len(results)}/1000 saved to {output_file}")
            
    except Exception as e:
        print(f"Error at iteration {i}: {e}")
        continue

print("✨ Done! Your full dataset is ready.")

Starting generation of 100 queries...
Error at iteration 35: too many values to unpack (expected 2)
Error at iteration 90: Cannot save file into a non-existent directory: 'dataset'
✨ Done! Your full dataset is ready.


In [27]:
import os

# Create the directory if it doesn't exist
if not os.path.exists("dataset"):
    os.makedirs("dataset")
    print("📁 Created 'dataset' directory.")
else:
    print("✅ 'dataset' directory already exists.")

# Now try saving again
df_100 = pd.DataFrame(results)
df_100.to_csv("dataset/queries_100.csv", index=False)
print(f"✅ Saved first {len(df_100)} queries to dataset/queries_100.csv")

📁 Created 'dataset' directory.
✅ Saved first 155 queries to dataset/queries_100.csv


In [28]:
df_100.to_csv("queries_100.csv", index=False)

In [29]:
import pandas as pd
import torch

# Settings for the marathon run
TOTAL_QUERIES = 1000
BATCH_SIZE = 5
output_file = "queries_1000.csv" # Saved directly to current directory
final_results = []

print(f"🚀 Starting the 1000-query generation...")

for i in range(0, TOTAL_QUERIES, BATCH_SIZE):
    # Rotate complexity levels: Simple -> Medium -> Complex
    level = ["Simple", "Medium", "Complex"][(i // BATCH_SIZE) % 3]
    
    try:
        # Calls the function we defined earlier
        batch = generate_gaia_queries(level, count=BATCH_SIZE)
        
        for entry in batch:
            if "|" in entry:
                sql, nl = entry.split("|", 1)
                final_results.append({
                    "complexity": level,
                    "sql": sql.replace("SQL:", "").strip(),
                    "question": nl.replace("NL:", "").strip()
                })
        
        # Save progress every 100 queries
        if len(final_results) % 100 == 0:
            pd.DataFrame(final_results).to_csv(output_file, index=False)
            print(f"📊 Progress checkpoint: {len(final_results)}/1000 queries saved.")

    except Exception as e:
        print(f"⚠️ Batch starting at {i} skipped due to error: {e}")
        continue

# Final save to ensure all data is captured
pd.DataFrame(final_results).to_csv(output_file, index=False)
print(f"✨ SUCCESS! 1000 queries generated and saved to {output_file}")

🚀 Starting the 1000-query generation...
📊 Progress checkpoint: 300/1000 queries saved.
✨ SUCCESS! 1000 queries generated and saved to queries_1000.csv


In [30]:
import os

file_path = "queries_1000.csv"
if os.path.exists(file_path):
    line_count = len(pd.read_csv(file_path))
    print(f"📈 The dataset currently has {line_count} queries stored.")
else:
    print("⏳ The first 100 queries haven't finished yet, so the file hasn't been created.")

📈 The dataset currently has 1332 queries stored.
